# DIFUSCO — TSP Benchmark

Standard TSP (no time windows, no disruptions).
Used as the reference benchmark for `docs/src/doc_difusco.tex`.

| Section | Content |
|---------|--------|
| 1 | Load model |
| 2 | Baseline helpers (NN, brute-force optimal) |
| 3 | Benchmark n=10 — NN vs DIFUSCO vs optimal |
| 4 | Benchmark n=50, 100 — NN vs DIFUSCO |
| 5 | DIFUSCO only — scalability across sizes |
| 6 | Loss curve |
| 7 | Tour visualisation |

In [6]:
import sys, os, time, glob, re, json as _json
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import permutations

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from data  import random_instance, tour_length, greedy_decode, optimal_tour
from model import DifuscoModel, MODEL_SIZES
from train import NoiseSchedule, sample

torch.manual_seed(42)
DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

SCALE = 200.0   # normalised [0,1]² coords × SCALE = minutes

# ── Dataset discovery ─────────────────────────────────────────────────────────
_HERE        = os.path.dirname(os.path.abspath('__file__'))
DATASETS_DIR = os.path.normpath(os.path.join(_HERE, '..', '..', 'datasets'))

def discover_datasets(datasets_dir=DATASETS_DIR, max_n=None):
    """Return sorted list of (n, path) for all tsptwd_n*.json files."""
    result = []
    for f in sorted(glob.glob(os.path.join(datasets_dir, 'tsptwd_n*.json'))):
        m = re.search(r'tsptwd_n(\d+)\.json$', f)
        if m:
            n = int(m.group(1))
            if max_n is None or n <= max_n:
                result.append((n, f))
    return sorted(result, key=lambda x: x[0])

def load_coords_from_json(path):
    """Load (x, y) from a tsptwd_n*.json — depot at index 0, then clients."""
    with open(path, encoding='utf-8') as fh:
        d = _json.load(fh)
    nodes = [d['depot']] + d['clients']
    return torch.tensor([[v['x'], v['y']] for v in nodes], dtype=torch.float32)

AVAILABLE = discover_datasets()
print(f'Found {len(AVAILABLE)} dataset(s):',
      ', '.join(f'n={n}' for n, _ in AVAILABLE))


Device: cpu
Found 9 dataset(s): n=5, n=10, n=50, n=100, n=200, n=300, n=500, n=1000, n=10000


## Section 1 — Load model

Train first if checkpoint does not exist:
```bash
python train.py --mode tsp --n 10 --epochs 3000 --size small
```

In [7]:
import torch._utils  # force-load submodule (fixes AttributeError on some installs)

SIZE = 'medium'
d, L, T = MODEL_SIZES[SIZE]

# TSP model: node_dim=2 (x,y), edge_dim=1 (distance only)
model_tsp = DifuscoModel(d=d, L=L, node_dim=2, edge_dim=1)
schedule  = NoiseSchedule(T=T)

ckpt = f'model/difusco_{SIZE}_tsp.pt'
if os.path.exists(ckpt):
    model_tsp.load_state_dict(torch.load(ckpt, map_location='cpu', weights_only=False))
    print(f'Loaded {ckpt}')
else:
    print(f'WARNING: {ckpt} not found — using random weights (results will be meaningless)')
model_tsp.eval()
print(f'Parameters: {sum(p.numel() for p in model_tsp.parameters()):,}')

Loaded model/difusco_medium_tsp.pt
Parameters: 668,679


## Section 2 — Baseline helpers

In [8]:
def run_difusco(coords, n_steps=50):
    """Single DIFUSCO sample → greedy decoded tour."""
    p = sample(model_tsp, coords, schedule, edge_extra=None,
               n_steps=n_steps, ddim=True, device=DEVICE)
    return greedy_decode(p.cpu(), start=0)


## Section 3 — Benchmark n=10 (with optimal comparison)

For n=10, brute-force optimal is fast (9! = 362 880 permutations).

In [9]:
n10_entry = next(((n, p) for n, p in AVAILABLE if n == 10), None)
if n10_entry is None:
    raise FileNotFoundError('datasets/tsptwd_n10.json not found.')

coords_10 = load_coords_from_json(n10_entry[1])
print(f'Loaded n=10 dataset: {coords_10.shape[0]} nodes')

_, opt_len_10  = optimal_tour(coords_10)
opt_scaled_10  = opt_len_10 * SCALE

t0             = time.perf_counter()
dif_tour_10    = run_difusco(coords_10)
dif_time_10    = (time.perf_counter() - t0) * 1000
dif_len_10     = tour_length(coords_10, dif_tour_10) * SCALE
gap_10         = (dif_len_10 / opt_scaled_10 - 1.0) * 100.0

print(f'n=10  (lengths in minutes)')
print(f'{"Method":<12} {"Length (min)":>14} {"Gap to opt":>12} {"Time (ms)":>12}')
print('-' * 53)
print(f'{"optimal":<12} {opt_scaled_10:>14.4f} {"-":>12} {"-":>12}')
print(f'{"difusco":<12} {dif_len_10:>14.4f} {gap_10:>+11.1f}% {dif_time_10:>11.1f}ms')


Loaded n=10 dataset: 11 nodes
n=10  (lengths in minutes)
Method         Length (min)   Gap to opt    Time (ms)
-----------------------------------------------------
optimal            478.7463            -            -
difusco            507.8151        +6.1%       189.0ms


## Section 4 — Benchmark n=50, n=100

In [10]:
all_results = {}   # {n: {'length': float, 'time_ms': float}}

for n, path in AVAILABLE:
    coords   = load_coords_from_json(path)
    actual_n = coords.shape[0]

    t0      = time.perf_counter()
    tour    = run_difusco(coords)
    time_ms = (time.perf_counter() - t0) * 1000
    length  = tour_length(coords, tour) * SCALE   # minutes

    all_results[n] = {'length': length, 'time_ms': time_ms}
    print(f'n={n:<6} ({actual_n} nodes)  DIFUSCO: {length:.4f} min  |  {time_ms:.1f} ms')


n=5      (6 nodes)  DIFUSCO: 592.4563 min  |  128.0 ms
n=10     (11 nodes)  DIFUSCO: 485.4428 min  |  121.0 ms
n=50     (51 nodes)  DIFUSCO: 1884.7559 min  |  351.3 ms
n=100    (101 nodes)  DIFUSCO: 2091.1579 min  |  826.5 ms
n=200    (201 nodes)  DIFUSCO: 2995.1857 min  |  5514.6 ms
n=300    (301 nodes)  DIFUSCO: 3435.1421 min  |  13403.4 ms
n=500    (501 nodes)  DIFUSCO: 4940.2737 min  |  37630.8 ms
n=1000   (1001 nodes)  DIFUSCO: 9019.7037 min  |  229587.0 ms


RuntimeError: [enforce fail at alloc_cpu.cpp:117] data. DefaultCPUAllocator: not enough memory: you tried to allocate 51210240512 bytes.

## Section 5 — DIFUSCO only (scalability)

In [ ]:
# Scalability results are now part of Section 4 (all_results dict).
# Re-run Section 4 to populate, then inspect:
for n, r in sorted(all_results.items()):
    print(f'n={n:<6}  {r["length"]:.4f} min  {r["time_ms"]:.1f} ms')


## Section 6 — Loss curve

In [ ]:
loss_path = f'model/difusco_{SIZE}_tsp_losses.npy'
if os.path.exists(loss_path):
    losses = np.load(loss_path)
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(losses, alpha=0.4, color='steelblue', lw=0.8, label='step loss')
    window = max(1, len(losses) // 100)
    smooth = np.convolve(losses, np.ones(window)/window, mode='valid')
    ax.plot(smooth, color='navy', lw=1.5, label=f'moving avg ({window})')
    ax.set(xlabel='Training step', ylabel='BCE loss', title=f'DIFUSCO TSP ({SIZE})')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('figures/benchmark/tsp_loss.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'Loss file not found: {loss_path}')

## Section 7 — Tour visualisation

In [ ]:
def plot_tour(coords, tour, ax, title='', color='steelblue'):
    n = coords.shape[0]
    for k in range(n):
        i, j = tour[k], tour[(k+1) % n]
        ax.annotate('', xy=coords[j], xytext=coords[i],
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.2))
    ax.scatter(coords[:, 0], coords[:, 1], c='white', edgecolors='black', s=60, zorder=5)
    for i, (x, y) in enumerate(coords):
        ax.text(x, y, str(i), ha='center', va='center', fontsize=7, zorder=6)
    ax.set(title=title, xticks=[], yticks=[])
    ax.set_aspect('equal')

coords = random_instance(12, seed=7)

tour_nn_v  = run_nn(coords)
tour_d_v   = run_difusco(coords)
tour_opt_v, opt_l = optimal_tour(coords)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
configs = [
    (tour_nn_v,  f'NN  ({tour_length(coords, tour_nn_v):.3f})',   'tomato'),
    (tour_d_v,   f'DIFUSCO ({tour_length(coords, tour_d_v):.3f})', 'steelblue'),
    (tour_opt_v, f'Optimal ({opt_l:.3f})',                         'seagreen'),
]
for ax, (tour, title, color) in zip(axes, configs):
    plot_tour(coords, tour, ax, title=title, color=color)
plt.suptitle('TSP n=12 — method comparison', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/benchmark/tsp_tour_comparison.png', dpi=150, bbox_inches='tight')
plt.show()